# Ready-to-React 학습 / 추론 워크스루

[Ready-to-React (ICLR 2025)](https://github.com/zju3dv/ready_to_react) 의 `ReactiveARDiffDecModel` 의 **학습 / 추론 흐름** 만 따라가며 공부하기 위한 파일입니다.

> 데이터셋이나 사전학습 가중치는 없습니다 → random init 상태로 **shape 만** 확인합니다. 학습/추론 코드 자체는 정상 동작합니다.

### 한 줄 요약

> 상대(B) 의 동작을 보고, 내(A) 의 **다음 4 프레임 동작** 을 실시간 생성한다.

### 모델 두 단계

```
[A 의 과거 자세] + [B 의 과거 자세]
        │
        ▼  (1) AR Diffusion           ─ 다음 4 프레임 분 latent 1개를 노이즈에서 디노이즈
   future_latent  (B, 1, 512)
        │
        ▼  (2) Online Motion Decoder  ─ latent 1개 → 실제 4 프레임 자세값으로 복원
   다음 4 프레임의 자세  (B, 4, 256)
```

### 자주 등장하는 차원 (한 번 외워두면 편합니다)

| 이름           | 끝 차원 | 무엇을 담나                                                                                        |
|----------------|--------|----------------------------------------------------------------------------------------------|
| `pose_series`  | **256** | 내(A) 의 자세 = 21 관절 자세(252) + 직전 프레임 root 기준 이동·방향(4)                              |
| `oppo_pose`    | **252** | 상대(B) 의 joint pos·rot·vel — **내(A) 의 root 좌표계 기준** 으로 표현 (논문 Eq.3 / `cal_oppo_pose`).<br>자기 root 신호 4 는 빠짐 |
| `root_info`    | **5**   | 디코더용 보조 root 신호 = 상대 root 기준 이동·방향(4) + 링 중심까지 거리(1)                            |
| `latents`      | **512** | VQ-VAE 가 *4 프레임 자세를 1 토큰* 으로 압축한 벡터                                                  |

`252 = 21 관절 × (위치 3 + 회전 6d + 속도 3)`  ← 한 관절이 12 차원

### 시간축 변수

| 이름      | 의미                                                |
|-----------|--------------------------------------------------|
| `B`       | batch 크기 (한 번에 처리하는 시퀀스 수)                  |
| `T`       | 프레임 수 (30 fps)                                    |
| `U` = 4   | **1 latent 가 담는 프레임 수**                           |
| `T_lat`   | latent 시퀀스 길이 (학습 시 `block_size = 15`)            |
| `T_pose`  | 프레임 시퀀스 길이 = `T_lat × U = 60`                    |

In [ ]:
# =====================================================================
# 준비 (학습/추론을 돌리기 위한 최소 setup)
#   - 한 번만 실행해 두면 아래 셀에서 model, B, T_lat, U, T_pose, dotdict 사용 가능
#   - 데이터셋·사전학습 가중치 없음 → norm 더미 + random init 으로 shape 만 확인
# =====================================================================
import os, sys
import numpy as np
import torch
from pathlib import Path

REPO = Path('/workspace/ready_to_react')   # 실행 환경에 맞게 수정
os.chdir(REPO); sys.path.insert(0, str(REPO))

# easyvolcap 은 import 시점에 sys.argv 를 읽어 cfg 를 만든다 → CLI 호출을 흉내냄
sys.argv = ['notebook', '-c', 'configs/exps/reactive_model.yaml', '-t', 'test']

# 데이터 통계 .npy 가 없으므로 mean=0, std=1 인 더미 생성 (shape 만 맞춤)
def _ensure_norm(path, feats):
    p = REPO / path
    if p.exists(): return
    p.parent.mkdir(parents=True, exist_ok=True)
    stats = np.zeros((2, feats), np.float32); stats[1] = 1.0
    np.save(p, stats)

_ensure_norm('data/boxing/reactive/motoken/pose_series.npy', 256)
_ensure_norm('data/boxing/reactive/react_motoken/oppo_pose.npy', 252)
_ensure_norm('data/boxing/reactive/react_motoken/root_info.npy', 5)

from easyvolcap.engine import cfg, MODELS
from easyvolcap.utils.base_utils import dotdict
from reactmotion.utils.engine_utils import discover_modules
from reactmotion.utils.motion_repr_transform import U      # = 4
discover_modules()

# 모델 빌드 (random init 상태)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MODELS.build(cfg.model_cfg).to(device).eval()

# 자주 쓰는 상수
B       = 2                              # batch 크기 (한 번에 처리하는 시퀀스 수)
T_lat   = cfg.window_cfg.block_size      # = 15  (latent 시퀀스 길이)
T_pose  = T_lat * U                      # = 60  (프레임 시퀀스 길이)

n_param = sum(p.numel() for p in model.parameters()) / 1e6
print(f'device : {device}')
print(f'model  : {type(model).__name__}  ({n_param:.1f} M params)')
print(f'U / T_lat / T_pose = {U} / {T_lat} / {T_pose}')

## 1. 학습 (Training)

> **학습 두 단계 중 어디?** (논문 Section 3.4)
> - **Stage 1** — VQ-VAE motion encoder/decoder + codebook 사전학습 (40k iter, EMA codebook)
> - **Stage 2** — VQ-VAE encoder 를 **고정** 한 채, next latent predictor + online motion decoder 를 **joint training** (40k iter, DDIM 50 step inference)
>
> 이 노트북이 따라가는 흐름은 **Stage 2** 입니다. 입력의 `latents (B, 15, 512)` 는 Stage 1 에서 미리 학습된 VQ-VAE 가 dequantize 해 만든 토큰이라고 가정합니다 (논문은 Stage 2 에서 ground truth `Ẑ_{f−1}` 를 그대로 사용한다고 명시).

학습은 한 윈도우(60 프레임 = 2 초) 의 시퀀스를 통째로 받고, **그 안의 14 개 시간 슬롯을 동시에** 학습합니다 (block_size 15 → 1 칸 밀어 다음 latent 를 예측하는 AR 방식).

### Step 1 — 학습 batch (DataLoader 가 만들어 주는 입력)

| key            | shape           | 자세히 보면                                                                         |
|----------------|----------------|-------------------------------------------------------------------------------|
| `latents`      | `(B, 15, 512)`  | VQ-VAE 가 60 프레임을 인코드한 latent 토큰 시퀀스. **15** = block_size, **512** = latent 차원 |
| `oppo_pose`    | `(B, 60, 252)`  | 상대(B) 의 60 프레임 joint pos·rot·vel (내 root 좌표계 기준)                                  |
| `pose_series`  | `(B, 60, 256)`  | 내(A) 의 60 프레임 자세 (252 + root 4)                                                     |
| `root_info`    | `(B, 60, 5)`    | 디코더용 보조 root 신호                                                                     |
| `mask`         | `(B, 15)`       | latent 시퀀스 padding (1 = 유효, 0 = pad)                                              |
| `pose_mask`    | `(B, 60)`       | 프레임 시퀀스 padding                                                                       |

> 같은 시퀀스인데 **latent 축은 15, 프레임 축은 60** — `U = 4` 배 차이입니다.

### Step 2 — `model.diffusion_forward(target, batch)` (디퓨전 학습)

`target = batch.latents (B, 15, 512)` 를 **시간으로 1 칸 밀어** AR 로 학습합니다.

```
target[:, :-1]  →  과거 latent 14 개   # 입력 (조건)
target[:,  1:]  →  다음 latent 14 개   # 정답 (x_start)
```

내부 흐름:

1. `oppo_pose` 를 latent 시간축에 맞추기 위해 **U = 4 stride 로 다운샘플** : `(B, 60, 252)` → `(B, 14, 252)` (`mocon_model.py` 의 `batch.oppo_pose[:, :-U:U]`)
2. `condition_forward(oppo_pose, past_latents)` 가 두 신호를 transformer encoder 로 합쳐 → `conditions (B, 14, 512)`
3. 정답 `x_start` 에 `q_sample` 로 가우시안 노이즈를 섞어 `x_t` 생성
4. 네트워크가 `x_t`, `timesteps`, `conditions` 를 보고 `x_start` 를 직접 예측 (`predict_xstart=True`)
5. `MSE(x_start_pred, x_start)` 손실 반환

> **함수명 풀이**
> - `q_sample` : 정답에 가우시안 노이즈를 섞어 `x_t` 만들기 (디퓨전의 forward process)
> - `predict_xstart=True` : 네트워크가 *노이즈* 가 아니라 *원래값* 자체를 예측하도록 학습
> - `condition_forward` : 상대 자세 + 과거 latent 를 합쳐 조건 벡터 만들기

### Step 3 — `model.decoder_forward(past_latents, future_latents, batch)` (디코더 학습)

디코더의 일은 **"직전 4 프레임 자세 + 직전 latent + 다음 latent → 다음 4 프레임 자세"** 입니다.
학습 시에는 14 개 슬롯을 한 번에 처리합니다.

**입력 reshape**
- `pose_series[:, :-U]` `(B, 56, 256)` → 4 프레임씩 잘라 `(B*14, 4, 256)`
- 각 latent 도 잘라 `(B*14, 1, 512)`

→ **14 개 시간 슬롯이 배치 차원에 합쳐져 병렬로** 학습됩니다.

**손실**
- 디코더가 예측한 다음 4 프레임 vs `pose_series[:, U:]` `(B, 56, 256)` (= 다음 4 프레임 정답)

> **shape 인덱스 풀이**
> - `(B, 14, 512)` 의 **14** = 한 윈도우 안에서 학습 가능한 슬롯 수 = `block_size − 1`
> - `(B, 56, 256)` 의 **56** = `14 슬롯 × 4 프레임`

In [ ]:
# 학습 batch 만들기 (DataLoader 가 만들어 주는 출력과 동일한 shape 의 fake batch)
batch = dotdict(
    latents     = torch.randn(B, T_lat,  512, device=device),  # (B, 15, 512)
    oppo_pose   = torch.randn(B, T_pose, 252, device=device),  # (B, 60, 252)
    pose_series = torch.randn(B, T_pose, 256, device=device),  # (B, 60, 256)
    root_info   = torch.randn(B, T_pose, 5,   device=device),  # (B, 60, 5)
    mask        = torch.ones (B, T_lat,        device=device), # (B, 15)
    pose_mask   = torch.ones (B, T_pose,       device=device), # (B, 60)
)
print('학습 batch shapes:')
for k, v in batch.items():
    print(f'  {k:<12s} {tuple(v.shape)}')

# Xnorm/Cnorm/Rnorm 으로 in-place 정규화 + train 모드 ON
model.train()
model.normalize_batch_data(batch)

target = batch.latents          # (B, 15, 512)

# ─── (1) 디퓨전 학습 ──────────────────────────────────────────────
# diffusion_forward 가 내부에서 알아서:
#   x_start    = target[:, 1:]     → (B, 14, 512)  '다음' latent (정답)
#   past_lat   = target[:, :-1]    → (B, 14, 512)  '과거' latent (조건)
#   oppo_pose  도 [:, :-U:U] 로 stride → (B, 14, 252)
diff_out, diff_loss = model.diffusion_forward(target, batch)
print('\n[diffusion_forward]')
print('  x_start 예측  :', tuple(diff_out.shape), '  # (B, 14, 512)')
print('  recon loss   :', float(diff_loss.detach()))

# ─── (2) 디코더 학습 ──────────────────────────────────────────────
# 14 슬롯을 배치 차원에 합쳐 (B*14, ...) 로 한 번에 처리
# 손실은 batch.pose_series[:, U:] (B, 56, 256) 과 비교
dec_pose_loss, dec_root_loss = model.decoder_forward(target[:, :-1], diff_out, batch)
print('\n[decoder_forward]')
print('  pose loss :', tuple(dec_pose_loss.shape), 'mean =', float(dec_pose_loss.mean()))
print('  root loss :', tuple(dec_root_loss.shape), 'mean =', float(dec_root_loss.mean()))

## 2. 추론 (Inference)

추론은 **매 step 마다 다음 4 프레임만** 예측해 결과를 자기 버퍼에 누적합니다 (streaming generation). 그래서 학습 batch 와 입력 shape 가 다릅니다.

### 학습 vs 추론 — 입력 shape 비교

```
                 학습              추론
oppo_pose        (B, 60, 252)      (B, T_lat, 252) *  ← * 아래 주석 참조
pose_series      (B, 60, 256)      (B,   4,   256)    ← 직전 4 프레임만
root_info        (B, 60,   5)      (B,   4,     5)
latents          (B, 15, 512)      (B, T_lat, 512)
```

> **\* `oppo_pose` 의 시간축은 누가 줄이는가?**
> 논문 개념상 상대 motion 은 frame-level 윈도우 `O_{i ∈ [f−W, f)}` 로 들어오고 history encoder 안에서 `d = 4` 로 downsample 됩니다.
> 실제 코드 (`mocon_model.py`) 를 보면 모드별로 다르게 처리됩니다:
> - **train 모드** : 모델이 `batch.oppo_pose[:, :-U:U]` 로 내부에서 stride
> - **eval 모드** : `batch.oppo_pose` 를 그대로 사용 → 호출자(streaming runner)가 `cal_oppo_pose(..., step=U)` 로 미리 stride 한 텐서를 넘긴다
>
> 즉 위 표의 추론 행에 적힌 `T_lat` 길이는 **이 코드의 추론 컨벤션** 한정입니다. "추론 입력은 원래부터 stride 된 형태" 라는 일반화된 명제는 아닙니다.

### `model.network_inference(batch)` — 추론 한 스텝

처리 흐름:

1. **조건 인코딩** — `condition_forward(oppo_pose, latents)` → `conditions (B, T_lat, 512)`
2. **DDIM 디노이즈 루프** — `ddim_sample_loop` 가 50 step 동안 노이즈에서 latent 시퀀스를 복원 → `future_latents (B, T_lat, 512)`
3. **마지막 1 토큰만 사용** — `future_latents[:, -1:]` `(B, 1, 512)` 가 *다음 4 프레임* 의 latent
4. **디코더로 자세 복원** — 직전 4 프레임 자세/root + 직전 latent + 새 latent → 다음 4 프레임 의 `pose_series` / `root_info`
5. **denormalize** — 정규화를 풀어 실제 좌표로 반환

> **함수명 풀이**
> - `condition_forward` : 조건(상대 자세 + 과거 latent) 을 transformer encoder 로 합쳐 조건 벡터 생성
> - `ddim_sample_loop` : 디퓨전의 *추론용* 디노이즈 반복. 학습의 1000 step 을 50 step 으로 빠르게 줄여 샘플
> - `network_inference` : 위 단계를 한 묶음으로 호출하는 모델의 진입점

> **shape 인덱스 풀이**
> - `pred_latents (B, 1, 512)` 의 **1** = 이번 스텝에 새로 만든 latent 토큰 1개
> - `pred_pose_series (B, 4, 256)` 의 **4** = 그 latent 가 풀어낸 다음 4 프레임

In [ ]:
# 데모용으로 DDIM 샘플 step 을 5 로 줄임 (원본 50 step)
model.eval()
model.skip_steps = model.diffusion_steps - 5
print('DDIM sample steps =', model.diffusion_steps - model.skip_steps)

# 추론 입력은 '현재 윈도우' 만 들어 있습니다
#   latents     (B, T_lat, 512)   지난 latent 시퀀스
#   oppo_pose   (B, T_lat, 252)   지난 상대 자세 (이미 U-stride 로 다운샘플된 상태)
#   pose_series (B,   U=4, 256)   직전 4 프레임의 내 자세
#   root_info   (B,   U=4,   5)   직전 4 프레임의 root 보조 신호
infer_batch = dotdict(
    latents     = torch.randn(B, T_lat, 512, device=device),
    oppo_pose   = torch.randn(B, T_lat, 252, device=device),
    pose_series = torch.randn(B, U,     256, device=device),
    root_info   = torch.randn(B, U,     5,   device=device),
)
print('infer_batch shapes:')
for k, v in infer_batch.items():
    print(f'  {k:<12s} {tuple(v.shape)}')

with torch.no_grad():
    pred_lat, pred_pose, pred_root = model.network_inference(infer_batch)

print('\n[network_inference] — 한 스텝의 출력')
print('  pred_latents     :', tuple(pred_lat.shape),  ' # (B, 1, 512) 다음 4 프레임 분 latent 1개')
print('  pred_pose_series :', tuple(pred_pose.shape), ' # (B, 4, 256) 다음 4 프레임 자세 (denorm 적용)')
print('  pred_root_info   :', tuple(pred_root.shape), ' # (B, 4,   5) 다음 4 프레임 root_info (denorm 적용)')

## 3. 추론 출력 자세히 보기 — `pose_series` 의 256 차원 분해

추론에서 나온 `pred_pose_series (B, 4, 256)` 의 **256** 차원이 실제로 무엇을 담는지 풀어 보면:

```
[ 0  : 252 ]  관절 자세 — 21 관절 × (pos 3 + rot6d 6 + vel 3)
[252 : 254 ]  root_off  (x, z)    ← 직전 프레임 *내* root 좌표계의 상대 이동
[254 : 256 ]  root_dir  (dx, dz)  ← 직전 프레임 *내* root 좌표계의 상대 방향
```

매 프레임이 **로컬 좌표계** 로 표현되기 때문에, 한 프레임씩 누적해 가며 AR 롤아웃이 가능합니다.

`MoReprTrans.split_pose(...)` 가 이 분해를 한 번에 해 줍니다 (회전 6d 는 자동으로 3×3 행렬로 변환).

In [12]:
from reactmotion.utils.motion_repr_transform import MoReprTrans

loc = MoReprTrans.split_pose(pred_pose)           # pred_pose: (B, U, 256)
for k, v in loc.items():
    print(f'{k:<10s} : {tuple(v.shape)}')

pose_pos   : (2, 4, 21, 3)
pose_rot   : (2, 4, 21, 3, 3)
pose_vel   : (2, 4, 21, 3)
root_off   : (2, 4, 2)
root_dir   : (2, 4, 2)
root_ctrl  : (2, 4, 4)


## 4. 정리

| 단계   | 함수                          | 입력 → 출력                                                          |
|--------|-------------------------------|--------------------------------------------------------------------|
| 학습   | `diffusion_forward`            | latent `(B,15,512)` → next-latent `(B,14,512)` 예측 + MSE loss      |
| 학습   | `decoder_forward`              | 14 슬롯 병렬 → pose `(B,56,256)` / root `(B,56,5)` 예측 + loss       |
| 추론   | `network_inference`            | 직전 4 프레임 + 지난 latent → 다음 4 프레임 pose / root              |

* **VQ-VAE** : motion **encoder + codebook** 으로 latent 표현을 만드는 데 사용 (Stage 1 에서 학습한 뒤 Stage 2 부터는 fix). 추론 시 latent → pose 복원은 VQ-VAE decoder 가 **아니라** **Online Motion Decoder** 가 담당합니다 — VQ-VAE decoder 는 시퀀스 전체가 모여야 풀 수 있어 streaming 에 부적합 (논문 ablation `w/o online decoder` 에서 FID 가 크게 악화).
* **Diffusion** : latent 공간에서 next-latent 1 개를 디노이즈로 만든다.
* **Decoder (Online Motion Decoder)** : 그 latent 1 개를 → 실제 4 프레임 자세값으로 복원해 streaming 생성을 가능케 한다 (직전 4 프레임 pose + root_info + 직전·현재 latent 를 입력).
* **Sparse-control 버전** (`SparseControlARDiffDecModel`) : 머리·양손 27 차원 `ctrl_info` 를 추가 조건으로 받음 (논문 Section 5).

random init 기준 모델 파라미터 수:

In [ ]:
def params(m): return sum(p.numel() for p in m.parameters()) / 1e6
print(f'ReactiveARDiffDecModel total : {params(model):6.2f} M')